In [ ]:
import os
import json
import time
from typing import List, Dict
from pinecone import Pinecone
from pinecone_text.sparse import BM25Encoder
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

def reload_rag_modules():
    import importlib
    import generator.generator as generator_module
    import retriever.retriever as retriever_module

    importlib.reload(generator_module)

    return generator_module.rag_chat, generator_module.clear_chat_history, retriever_module.initialize_reranker

rag_chat, clear_chat_history,initialize_reranker = reload_rag_modules()

c:\Users\thebi\Desktop\DocsGuide\src\projenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# EVALUATION CONFIGURATIO
# Retrieval Configuration
N_RETRIEVAL = 7 
N_GENERATION = 3
ALPHA = 0.5

# QA File Configuration
QA_FOLDER = "../data/qa"
QA_FILES = ["citizenship_qa","passport_qa"]

# Output Configuration
OUTPUT_FOLDER = "./evaluation_results/pure_hybrid"

# Rate Limiting
DELAY_SECONDS = 15  # Delay between API calls to avoid rate limits

# Pinecone Configuration
INDEX_NAME = "nepali-docs-hybrid"

# Model Configuration
DENSE_MODEL = "universalml/Nepali_Embedding_Model"
BM25_PARAMS_PATH = "embeddings/bm25_params.json"

print("=" * 80)
print("EVALUATION CONFIGURATION")
print("=" * 80)
print(f"Retrieval: Top-{N_RETRIEVAL} chunks (for evaluation metrics)")
print(f"Generation: Top-{N_GENERATION} chunks (for LLM answer)")
print(f"Alpha: {ALPHA} (0=BM25, 1=Dense)")
print(f"QA Files: {QA_FILES}")
print(f"Output Folder: {OUTPUT_FOLDER}")
print("=" * 80 + "\n")


EVALUATION CONFIGURATION
Retrieval: Top-7 chunks (for evaluation metrics)
Generation: Top-3 chunks (for LLM answer)
Alpha: 1 (0=BM25, 1=Dense)
QA Files: ['citizenship_qa', 'passport_qa']
Output Folder: ./evaluation_results/pure_dense



In [ ]:
def load_qa_file(folder_path: str, qa_type: str) -> Dict:
    """Load QA data from JSON file."""
    file_path = os.path.join(folder_path, f"{qa_type}.json")
    if not os.path.exists(file_path):
        print(f"⚠️ File not found: {file_path}")
        return None

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            if not isinstance(data, list):
                print(f"⚠️ Unexpected structure in {file_path} – expected list, got {type(data)}")
                return None
            return data
    except json.JSONDecodeError as e:
        print(f"❌ JSON decode error in {file_path}: {e}")
        return None


In [ ]:
import re

def calculate_answer_score(generated_answer: str, ground_truth_answer: str) -> float:
    prompt_text = """
You are an expert evaluator for a question-answering system about Nepali government documents.

**Task:** Compare the generated answer with the ground truth answer and assign a score from 0 to 1.

**Scoring Criteria:**
- 1.0: Perfect match - all key information is present and accurate
- 0.8-0.9: Very good - most key information is present with minor omissions
- 0.6-0.7: Good - main points covered but missing some details
- 0.4-0.5: Partial - some correct information but significant gaps
- 0.3-0.2: Poor - very limited correct information
- 0.0-0.1: Wrong or no useful information

**Ground Truth Answer:**
{ground_truth}

**Generated Answer:**
{generated}

**Instructions:**
1. Compare semantic meaning, not exact wording
2. Consider completeness and accuracy
3. Language doesn't matter (Nepali vs English) - focus on content
4. Output ONLY a single number between 0.0 and 1.0

**Score:**
"""
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)
    prompt = PromptTemplate(
        input_variables=["ground_truth", "generated"],
        template=prompt_text
    )
    
    try:
        response = llm.invoke(prompt.format(
            ground_truth=ground_truth_answer,
            generated=generated_answer
        ))
        
        # Extract score from response
        score_text = response.content.strip()
        numbers = re.findall(r'0?\.\d+|[01]\.0', score_text)
        if numbers:
            score = float(numbers[0])
            return max(0.0, min(1.0, score))
        else:
            print(f"⚠️ Could not parse score from: {score_text}")
            return 0.5
            
    except Exception as e:
        print(f"⚠️ Error calculating answer score: {e}")
        return 0.5 


def calculate_retrieval_metrics(retrieved_ids: List[str], relevant_ids: List[int]) -> Dict[str, float]:
    # Convert retrieved IDs to integers for comparison
    try:
        retrieved_set = set([int(rid.split('_')[-1]) if '_' in rid else int(rid) for rid in retrieved_ids if rid])
    except:
        retrieved_set = set()
    
    relevant_set = set(relevant_ids)
    
    if len(retrieved_set) == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    
    if len(relevant_set) == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}
    
    # Calculate metrics
    true_positives = len(retrieved_set & relevant_set)
    
    precision = true_positives / len(retrieved_set) if len(retrieved_set) > 0 else 0.0
    recall = true_positives / len(relevant_set) if len(relevant_set) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
def evaluate_single_question(
    index,
    dense_embeddings,
    bm25_encoder: BM25Encoder,
    question: str,
    ground_truth_answer: str,
    relevant_chunks: List[int],
    alpha: float = 0.5,
    n_retrieval: int = 5,
    n_generation: int = 3
) -> Dict:
    """
    Evaluate a single question.
    
    Args:
        index: Pinecone index
        dense_embeddings: Dense embedding model
        bm25_encoder: BM25 encoder
        question: Question to evaluate
        ground_truth_answer: Ground truth answer
        relevant_chunks: List of relevant chunk IDs
        alpha: Hybrid search parameter
        n_retrieval: Number of chunks to retrieve for evaluation
        n_generation: Number of chunks to use for generation
        
    Returns:
        Dict with evaluation results
    """
    print(f"\n🔍 Evaluating: {question[:80]}...")
    start = time.time()
    # Get RAG response
    result = rag_chat(
        index=index,
        query=question,
        dense_embeddings=dense_embeddings,
        bm25_encoder=bm25_encoder,
        alpha=alpha,
        n_retrieval=n_retrieval,
        n_generation=n_generation 
    )
    elapsed = time.time() - start
    generated_answer = result.get("answer", "")
    retrieved_chunk_ids = result.get("retrieved_chunk_ids", [])
    sources = result.get("sources", [])

    answer_score = calculate_answer_score(generated_answer, ground_truth_answer)
    
    # Calculate retrieval metrics (based on all retrieved chunks)
    retrieval_metrics = calculate_retrieval_metrics(retrieved_chunk_ids, relevant_chunks)
    
    print(f"   Answer Score: {answer_score:.3f}")
    print(f"   Precision@{n_retrieval}: {retrieval_metrics['precision']:.3f}")
    print(f"   Recall@{n_retrieval}: {retrieval_metrics['recall']:.3f}")
    print(f"   F1@{n_retrieval}: {retrieval_metrics['f1']:.3f}")
    
    return {
        "question": question,
        "ground_truth_answer": ground_truth_answer,
        "generated_answer": generated_answer,
        "retrieved_chunk_ids": retrieved_chunk_ids,
        "relevant_chunk_ids": relevant_chunks,
        "reranked_chunk_ids": result.get("reranked_chunk_ids", []),
        "response_time_sec": round(elapsed, 3),
        "sources": sources,
        "answer_score": answer_score,
        "precision": retrieval_metrics["precision"],
        "recall": retrieval_metrics["recall"],
        "f1": retrieval_metrics["f1"]
    }


In [ ]:
def run_evaluation(
    index,
    dense_embeddings,
    bm25_encoder: BM25Encoder,
    qa_folder: str,
    qa_files: List[str],
    output_folder: str = "./evaluation_results",
    alpha: float = 0.5,
    n_retrieval: int = 5,
    n_generation: int = 3,
    delay_seconds: int = 2
):
    """
    Run evaluation on all QA files.
    
    Args:
        index: Pinecone index
        dense_embeddings: Dense embedding model
        bm25_encoder: BM25 encoder
        qa_folder: Path to folder containing QA JSON files
        qa_files: List of QA file names (without .json extension)
        output_folder: Where to save evaluation results
        alpha: Hybrid search alpha parameter
        n_retrieval: Number of chunks to retrieve for evaluation
        n_generation: Number of chunks to use for generation
        delay_seconds: Delay between API calls
    """
    os.makedirs(output_folder, exist_ok=True)
    
    overall_results = {
        "config": {
            "n_retrieval": n_retrieval,
            "n_generation": n_generation,
            "alpha": alpha
        },
        "total_questions": 0,
        "avg_answer_score": 0.0,
        "avg_precision": 0.0,
        "avg_recall": 0.0,
        "avg_f1": 0.0,
        "by_document_type": {}
    }
    
    for qa_type in qa_files:
        print(f"\n{'='*80}")
        print(f"Evaluating: {qa_type}")
        print(f"{'='*80}")
        
        # Load QA data
        qa_data = load_qa_file(qa_folder, qa_type)
        if qa_data is None:
            continue
        
        # Clear chat history before each document type
        clear_chat_history()
        
        results_per_file = []
        metrics_sum = {
            "answer_score": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0
        }
        
        # Evaluate each question
        for i, qa_pair in enumerate(qa_data, 1):
            question = qa_pair.get("question", "")
            ground_truth = qa_pair.get("answer", "")
            relevant_chunks = qa_pair.get("relevant_chunks", [])
            
            if not question or not ground_truth:
                print(f"⚠️ Skipping invalid QA pair at index {i}")
                continue
            
            print(f"\n[{qa_type}] Question {i}/{len(qa_data)}")
            
            # Evaluate
            result = evaluate_single_question(
                index=index,
                dense_embeddings=dense_embeddings,
                bm25_encoder=bm25_encoder,
                question=question,
                ground_truth_answer=ground_truth,
                relevant_chunks=relevant_chunks,
                alpha=alpha,
                n_retrieval=n_retrieval,
                n_generation=n_generation
            )
            
            results_per_file.append(result)
            
            # Accumulate metrics
            metrics_sum["answer_score"] += result["answer_score"]
            metrics_sum["precision"] += result["precision"]
            metrics_sum["recall"] += result["recall"]
            metrics_sum["f1"] += result["f1"]
            
            # Delay to avoid rate limits
            time.sleep(delay_seconds)
        
        # Calculate averages for this document type
        num_questions = len(results_per_file)
        if num_questions > 0:
            avg_metrics = {
                "num_questions": num_questions,
                "avg_answer_score": metrics_sum["answer_score"] / num_questions,
                "avg_precision": metrics_sum["precision"] / num_questions,
                "avg_recall": metrics_sum["recall"] / num_questions,
                "avg_f1": metrics_sum["f1"] / num_questions
            }
            
            overall_results["by_document_type"][qa_type] = avg_metrics
            overall_results["total_questions"] += num_questions
            
            print(f"\n{'='*80}")
            print(f"Results for {qa_type}:")
            print(f"  Questions Evaluated: {num_questions}")
            print(f"  Avg Answer Score: {avg_metrics['avg_answer_score']:.3f}")
            print(f"  Avg Precision@{n_retrieval}: {avg_metrics['avg_precision']:.3f}")
            print(f"  Avg Recall@{n_retrieval}: {avg_metrics['avg_recall']:.3f}")
            print(f"  Avg F1@{n_retrieval}: {avg_metrics['avg_f1']:.3f}")
            print("=" * 80)
        
        # Save detailed results for this file
        output_file = os.path.join(output_folder, f"eval_{qa_type}.json")
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump({
                "document_type": qa_type,
                "config": {
                    "n_retrieval": n_retrieval,
                    "n_generation": n_generation,
                    "alpha": alpha
                },
                "metrics": avg_metrics if num_questions > 0 else {},
                "detailed_results": results_per_file
            }, f, ensure_ascii=False, indent=2)
        
        print(f"✅ Saved detailed results to {output_file}")
    
    # Calculate overall averages
    if overall_results["total_questions"] > 0:
        for doc_type, metrics in overall_results["by_document_type"].items():
            weight = metrics["num_questions"] / overall_results["total_questions"]
            overall_results["avg_answer_score"] += metrics["avg_answer_score"] * weight
            overall_results["avg_precision"] += metrics["avg_precision"] * weight
            overall_results["avg_recall"] += metrics["avg_recall"] * weight
            overall_results["avg_f1"] += metrics["avg_f1"] * weight
    
    # Save overall summary
    summary_file = os.path.join(output_folder, "evaluation_summary.json")
    with open(summary_file, "w", encoding="utf-8") as f:
        json.dump(overall_results, f, ensure_ascii=False, indent=2)
    
    print(f"\n{'='*80}")
    print("OVERALL EVALUATION SUMMARY")
    print("=" * 80)
    print(f"Configuration:")
    print(f"  - Retrieved {n_retrieval} chunks for evaluation")
    print(f"  - Used top {n_generation} chunks for generation")
    print(f"  - Alpha: {alpha}")
    print(f"\nResults:")
    print(f"  Total Questions: {overall_results['total_questions']}")
    print(f"  Overall Avg Answer Score: {overall_results['avg_answer_score']:.3f}")
    print(f"  Overall Avg Precision@{n_retrieval}: {overall_results['avg_precision']:.3f}")
    print(f"  Overall Avg Recall@{n_retrieval}: {overall_results['avg_recall']:.3f}")
    print(f"  Overall Avg F1@{n_retrieval}: {overall_results['avg_f1']:.3f}")
    print("=" * 80)
    print(f"\n✅ Saved summary to {summary_file}")
    
    return overall_results


In [ ]:
#Main Evaluation Pipeline
# Get API key
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    raise ValueError("Please set the PINECONE_API_KEY environment variable!")

# Initialize Pinecone
print("Initializing Pinecone...")
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)
print(f"✅ Connected to Pinecone index: {INDEX_NAME}")

# Load dense embeddings
print("🔧 Loading dense embeddings...")
dense_embeddings = HuggingFaceEmbeddings(model_name=DENSE_MODEL)
print("✅ Dense embeddings loaded")

# Load BM25 encoder
print("Loading BM25 encoder...")
bm25_encoder = BM25Encoder()
if os.path.exists(BM25_PARAMS_PATH):
    bm25_encoder.load(BM25_PARAMS_PATH)
    print("✅ BM25 encoder loaded")
else:
    print(f"⚠️ BM25 parameters not found at {BM25_PARAMS_PATH}")
    raise Exception("BM25 parameters not found")

initialize_reranker()
# Run evaluation
print("\n🚀 Starting evaluation pipeline...\n")
results = run_evaluation(
    index=index,
    dense_embeddings=dense_embeddings,
    bm25_encoder=bm25_encoder,
    qa_folder=QA_FOLDER,
    qa_files=QA_FILES,
    output_folder=OUTPUT_FOLDER,
    alpha=ALPHA,
    n_retrieval=N_RETRIEVAL,
    n_generation=N_GENERATION,
    delay_seconds=DELAY_SECONDS
)

print("\n✅ Evaluation complete!")


Initializing Pinecone...
✅ Connected to Pinecone index: nepali-docs-hybrid
🔧 Loading dense embeddings...
✅ Dense embeddings loaded
Loading BM25 encoder...
✅ BM25 encoder loaded
Loading Jina reranker model...


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention impl

Reranker model loaded successfully on cuda!

🚀 Starting evaluation pipeline...


Evaluating: citizenship_qa
🗑️ Chat history cleared!

[citizenship_qa] Question 1/18

🔍 Evaluating: नेपाली नागरिकताको प्रमाणपत्र लिनका लागि कुन कार्यालयमा जानुपर्छ?...

Original Query: नेपाली नागरिकताको प्रमाणपत्र लिनका लागि कुन कार्यालयमा जानुपर्छ?
Language: nepali
Rewritten Query: नेपाली नागरिकताको प्रमाणपत्र लिनका लागि कुन कार्यालयमा जानुपर्छ?
Document Type: citizenship
Category Tag: procedure

Generating dense vector...
Performing DENSE-ONLY search for top-7 chunks...
Retrieved 7 chunks

Retrieved 7 chunks for evaluation
Using top 3 chunks for answer generation

Final context chunks prepared for RAG prompt:
--- Chunk 1 ---
Chunk ID: 45
Source Link: https://www.moha.gov.np/en/page/citizenship-10
Source Type: Citizenship_Faqs
Preview: Base Chunk:
नागरिकता लिन जानुपर्ने कार्यालयहरू र तिनीहरूका कार्यहरू: नागरिकता सम्बन्धी कामको लागि वडा कार्यालय, स्थानीय कार्यकारिणीको कार्यालय, र प्रमुख जिल्ला अधिकारी (सीडी